<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/preprocessing/split_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
data = []
with open('train_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [ ]:
result_list = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']
    mentions = []
    for label in entry['annotations']:
        label_values = list(label.values())
        mentions.append(label_values)
    result_list.append({
            'patient_id': patient_id,
            'label': mentions,
            'text': text
        })

In [ ]:
import pandas as pd
result_df = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']

    for annotation in entry['annotations']:
        result_df.append({
            'patient_id': patient_id,
            'start': annotation['start'],
            'end': annotation['end'],
            'code': annotation['code'],
            'mention': annotation['mention']
        })
result_df = pd.DataFrame(result_df)
print(result_df)

       patient_id  start   end code                mention
0               3    220   226  I21                 NSTEMI
1               3    530   544  Y84         στεφανιογραφία
2               3    549   563  Z95         αγγειοπλαστική
3               3   1003  1017  Y84         Στεφανιογραφία
4               3   1020  1034  Z95         Αγγειοπλαστική
...           ...    ...   ...  ...                    ...
10163        6559    425   428  I21                    ΟΕΜ
10164        6559    555   568  E78          Δυσλιπιδαιμία
10165        6559    655   669  I30         περικαρδίτιδος
10166        6559    730   751  I30  οξείας περικαρδίτιδας
10167        6559   1037  1054  R07      Προκαρδίου Άλγους

[10168 rows x 5 columns]


In [ ]:
print(f"Total Annotations in the Dataset: {len(result_df)}")

patient_stats = result_df.groupby('patient_id').size().agg(['count', 'min', 'max', 'mean'])

print("Number of Patients/ EHR's:", patient_stats['count'])
print("Min Annotations per Patient:", patient_stats['min'])
print("Max Annotations per Patient:", patient_stats['max'])
print("Average Annotations per Patient:", patient_stats['mean'])

Total Annotations in the Dataset: 10168
Number of Patients/ EHR's: 1000.0
Min Annotations per Patient: 1.0
Max Annotations per Patient: 33.0
Average Annotations per Patient: 10.168


In [ ]:
j = 0
annotations_new = []
b_entity_count = 0
i_entity_count = 0
for i, document_data in enumerate(result_list):
    patient_id = document_data["patient_id"]
    text = document_data["text"]
    recognized_entities = []
    previous_end = -1
    for annotation in result_list[i]["label"]:
        start = annotation[0]
        end = annotation[1]
        icd_code = annotation[2]
        text1 = text
        name = text1[start:end]
        words = name.split()
        current_start = start
        for idx, word in enumerate(words):
            if idx == 0:
                type = "B-ENTITY"
                recognized_entities.append({
                "start": start,
                "end": start + len(word),
                "type": type,
                'word_name': word,
                'icd_code': icd_code })
                current_start = start + len(word)
                b_entity_count = b_entity_count + 1
        if current_start < end:
          type = "I-ENTITY"
          recognized_entities.append({
          "start": current_start,
          "end": end,
          "type": type,
          'word_name': word,
          'icd_code': icd_code })
          i_entity_count = i_entity_count + 1
          current_start = current_start + len(word)

    document_entry = {
        "patient_id": patient_id,
        "text": text,
        "recognized_entities": recognized_entities
    }
    annotations_new.append(document_entry)
print("Number of B entities:", b_entity_count)
print("Number of I entities:", i_entity_count)

Number of B entities: 10168
Number of I entities: 5557


In [ ]:
dict = {}
for patient in annotations_new:
    codes = []
    for entity in patient['recognized_entities']:
      if entity['icd_code'] not in codes:
         codes.append(entity['icd_code'])
    dict[patient['patient_id']] = codes
print(dict)

{3: ['I21', 'Y84', 'Z95'], 4: ['R00', 'I48', 'I10', 'I79'], 11: ['R06', 'I49', 'I10', 'I26'], 14: ['R06', 'R55', 'I48', 'I35', 'Z95', 'I50', 'E78', 'J81', 'I08'], 23: ['R55', 'I25', 'I21', 'Z95', 'I35', 'I36'], 26: ['R06', 'I48', 'I35', 'E11', 'K92', 'J81'], 35: ['I48', 'I10', 'N18', 'E11', 'E03', 'J96', 'I50', 'N17', 'R60', 'I07', 'I34', 'I37', 'I35', 'I31'], 39: ['R00', 'R42', 'R53', 'N18', 'E11', 'E78', 'I48', 'E03', 'I10', 'R60', 'Z95', 'N17'], 43: ['R07', 'I25', 'Z95', 'Y84'], 46: ['R07', 'R06', 'R11', 'I48', 'I10', 'I21', 'I49', 'Z95', 'I34', 'I36'], 51: ['I50', 'I21', 'I48', 'I25'], 62: ['R07', 'R11', 'R61', 'E13', 'D56', 'Z95', 'I21', 'I34'], 63: ['R07', 'R11', 'R20', 'I25', 'Z95'], 65: ['R42', 'R53', 'R00', 'I48', 'I10', 'G45', 'I45', 'I07'], 73: ['R07', 'Y84', 'I40', 'I36'], 77: ['I50', 'R60', 'I48', 'I21', 'I47', 'R03', 'I08'], 89: ['R06', 'I48', 'I10', 'I71', 'I79', 'I27', 'I51', 'I31', 'J90'], 102: ['R06', 'R60', 'J90', 'I48', 'Z95', 'I25', 'I70', 'I50', 'I35', 'I08'], 121

In [ ]:
with open('codes_list.txt', 'r') as file:
    code_list = file.read().splitlines()
print(code_list)

['R60', 'J81', 'R00', 'N18', 'E78', 'I49', 'R55', 'I31', 'J90', 'I45', 'I20', 'I47', 'I22', 'R42', 'J44', 'R50', 'R53', 'I08', 'E03', 'N17', 'R61', 'R11', 'D64', 'I42', 'I63', 'R10', 'I36', 'I24', 'I71', 'Z82', 'I27', 'I05', 'I46', 'J98', 'E13', 'I64', 'I72', 'I26', 'J45']


In [ ]:
result_df['code']
by_patient_df = result_df.groupby('patient_id')['code'].apply(list).reset_index()
print(by_patient_df)

     patient_id                                               code
0             3                [I21, Y84, Z95, Y84, Z95, I21, Z95]
1             4      [R00, I48, I48, I48, I10, R00, I79, R00, I79]
2            11                     [R06, I49, I49, I10, I26, I26]
3            14  [R06, R55, I48, I48, I35, Z95, I50, I35, I50, ...
4            23  [R55, I25, I21, Z95, Z95, R55, Z95, Z95, I25, ...
..          ...                                                ...
995        6540  [R06, R60, I48, R60, J90, I48, E03, C18, R60, ...
996        6547                          [I44, E78, I44, Z95, I44]
997        6551                                    [R06, R53, I50]
998        6552                               [R07, R06, Y84, R07]
999        6559                     [R07, I21, E78, I30, I30, R07]

[1000 rows x 2 columns]


In [ ]:
from collections import defaultdict

code_patient_map = defaultdict(set)

for _, row in by_patient_df.iterrows():
    patient_id = row['patient_id']
    codes = set(row['code'])
    for code in codes:
        code_patient_map[code].add(patient_id)

code_patient_counts = {code: len(patients) for code, patients in code_patient_map.items()}

import pandas as pd
code_counts_df = pd.DataFrame.from_dict(code_patient_counts, orient='index', columns=['patient_count'])
code_counts_df = code_counts_df.sort_values(by='patient_count', ascending=False)

print(code_counts_df)


     patient_count
Z95            514
I10            455
I25            408
R07            375
I48            339
..             ...
R57              1
C05              1
C32              1
I39              1
C80              1

[144 rows x 1 columns]


In [ ]:
# Reset index so 'code' becomes a column
code_counts_df = code_counts_df.reset_index().rename(columns={'index': 'code'})

# Print the full DataFrame without truncation
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print("\nAll codes by number of unique patients:\n")
    print(code_counts_df.to_string(index=False))


All codes by number of unique patients:

code  patient_count
 Z95            514
 I10            455
 I25            408
 R07            375
 I48            339
 I21            335
 R06            316
 Y84            307
 E11            288
 I50            257
 I34            242
 I07            192
 I35            154
 I44            150
 R60            140
 E78            139
 R00            136
 J81            120
 N18            109
 I49             87
 R55             82
 I20             76
 J90             74
 R53             72
 R42             71
 I45             70
 J44             67
 R50             63
 I08             55
 I22             51
 I47             50
 R61             47
 R11             46
 I31             46
 E03             46
 N17             41
 I63             34
 Z82             30
 I24             30
 I36             30
 D64             30
 R10             29
 I42             27
 I71             25
 E13             24
 I27             23
 I64             2

In [ ]:
len(code_list)
code_list

['R60',
 'J81',
 'R00',
 'N18',
 'E78',
 'I49',
 'R55',
 'I31',
 'J90',
 'I45',
 'I20',
 'I47',
 'I22',
 'R42',
 'J44',
 'R50',
 'R53',
 'I08',
 'E03',
 'N17',
 'R61',
 'R11',
 'D64',
 'I42',
 'I63',
 'R10',
 'I36',
 'I24',
 'I71',
 'Z82',
 'I27',
 'I05',
 'I46',
 'J98',
 'E13',
 'I64',
 'I72',
 'I26',
 'J45']

In [ ]:
 for i in range(0,len(code_list)-1):
    by_patient_df['has_target_code'] = by_patient_df['code'].apply(
        lambda codes: any(code in code_list[i] for code in codes))
    print(by_patient_df['has_target_code'].value_counts())

has_target_code
False    860
True     140
Name: count, dtype: int64
has_target_code
False    880
True     120
Name: count, dtype: int64
has_target_code
False    864
True     136
Name: count, dtype: int64
has_target_code
False    891
True     109
Name: count, dtype: int64
has_target_code
False    861
True     139
Name: count, dtype: int64
has_target_code
False    913
True      87
Name: count, dtype: int64
has_target_code
False    918
True      82
Name: count, dtype: int64
has_target_code
False    954
True      46
Name: count, dtype: int64
has_target_code
False    926
True      74
Name: count, dtype: int64
has_target_code
False    930
True      70
Name: count, dtype: int64
has_target_code
False    924
True      76
Name: count, dtype: int64
has_target_code
False    950
True      50
Name: count, dtype: int64
has_target_code
False    949
True      51
Name: count, dtype: int64
has_target_code
False    929
True      71
Name: count, dtype: int64
has_target_code
False    933
True      67
Name: 

In [ ]:
%pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [ ]:
from datasets import Dataset, Features, Sequence, Value, ClassLabel
tokens_list = []
ner_tags_list = []
patient_id_list = []
for j,dat in enumerate(annotations_new):
    tokens = (list(dat["text"]))
    ner_tags = ["O"] * len(tokens)
    for ent in annotations_new[j]["recognized_entities"]:
        for i in range(ent['start'], ent['end']):
            ner_tags[i] = ent['type']
    tokens_list.append(tokens)
    ner_tags_list.append(ner_tags)
    patient_id_list.append(dat['patient_id'])

features = Features({
    "tokens": Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
    "ner_tags": Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None),
    "patient_id": Value(dtype='int64', id=None)
})
ds = Dataset.from_dict(
    {"tokens": tokens_list, "ner_tags": ner_tags_list,"patient_id": patient_id_list},
    features=features
)

In [ ]:
from sklearn.model_selection import train_test_split
random_seed = 42
total_size = len(ds)
train_size = int(total_size * 0.8)
val_size = total_size-train_size

print(f"Total Size: {total_size}")
print(f"Train Size: {train_size}")
print(f"Validation Size: {val_size}")

patient_ids = ds['patient_id']
indices = range(len(patient_ids))

patient_ids = ds['patient_id']
patient_ids_train, patient_ids_val = train_test_split(patient_ids, test_size=val_size, random_state=random_seed)

ds_train_data = [obj for obj in ds if obj['patient_id'] in patient_ids_train]
ds_val_data = [obj for obj in ds if obj['patient_id'] in patient_ids_val]

ds_train = Dataset.from_dict({"tokens": [obj['tokens'] for obj in ds_train_data],
                              "ner_tags": [obj['ner_tags'] for obj in ds_train_data],
                              "patient_id": [obj['patient_id'] for obj in ds_train_data]})
ds_val = Dataset.from_dict({"tokens": [obj['tokens'] for obj in ds_val_data],
                            "ner_tags": [obj['ner_tags'] for obj in ds_val_data],
                            "patient_id": [obj['patient_id'] for obj in ds_val_data]})

final_dataset = {
    'train': ds_train,
    'validation': ds_val
}
print('Final dataset structure:')
final_dataset

Total Size: 1000
Train Size: 800
Validation Size: 200
Final dataset structure:


{'train': Dataset({
     features: ['tokens', 'ner_tags', 'patient_id'],
     num_rows: 800
 }),
 'validation': Dataset({
     features: ['tokens', 'ner_tags', 'patient_id'],
     num_rows: 200
 })}

In [ ]:
import pickle
with open('final_dataset.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)